In [ ]:
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MetroTest")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session started successfully!")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, hour, dayofweek, when, count, mean, udf
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.recommendation import ALS
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import mean as _mean, variance as _variance, stddev as _stddev
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.evaluation import ClusteringEvaluator
import numpy as np

# Initialize Spark Session
print("\n==============================")
print("Initializing Spark Session")
print("==============================")
spark = SparkSession.builder.appName("DelhiMetroBigDataAnalytics").getOrCreate()

# Load Data
print("\n==============================")
print("Loading CSV Data...")
print("==============================")
df = spark.read.csv("metro.csv", header=True, inferSchema=True)
print("Data Loaded Successfully.")
print("Total Records:", df.count())
print("Columns:", df.columns)

# Data Preprocessing & Cleansing
print("\n==============================")
print("Data Preprocessing & Cleansing")
print("==============================")
df = df.withColumn("entry_time", col("entry_time").cast("timestamp")) \
       .withColumn("exit_time", col("exit_time").cast("timestamp")) \
       .withColumn("travel_duration_minutes", col("travel_duration_minutes").cast(IntegerType())) \
       .withColumn("fare_amount", col("fare_amount").cast(IntegerType())) \
       .withColumn("hour_of_day", hour(col("entry_time"))) \
       .withColumn("day_of_week", dayofweek(col("entry_time"))) \
       .withColumn("is_peak_hour", when(col("hour_of_day").isin([7,8,9,17,18,19,20]), 1).otherwise(0)) \
       .withColumn("is_weekend", when(col("day_of_week").isin([1,7]), 1).otherwise(0))

print("\nData Schema:")
df.printSchema()
print("\nSample Data:")
df.show(5, truncate=False)

# Entity Resolution
print("\n==============================")
print("Normalizing Station Names")
print("==============================")
df = df.withColumn("entry_station_norm", lower(trim(col("entry_station")))) \
       .withColumn("exit_station_norm", lower(trim(col("exit_station"))))
print("Station names normalized.")

print("\n==============================")
print("Building ALS Recommendation System")
print("==============================")

ratings = df.select(
    col("card_id").alias("user"),
    col("entry_station_norm").alias("item"),
    col("fare_amount").alias("rating")
).distinct()

user_indexer = StringIndexer(inputCol="user", outputCol="userIndex", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="item", outputCol="itemIndex", handleInvalid="skip")
als = ALS(userCol="userIndex", itemCol="itemIndex", ratingCol="rating", coldStartStrategy="drop")

rec_pipeline = Pipeline(stages=[user_indexer, item_indexer, als])
train_ratings, test_ratings = ratings.randomSplit([0.8, 0.2], seed=42)
rec_model = rec_pipeline.fit(train_ratings)
predictions = rec_model.transform(test_ratings)

reg_evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse_value = reg_evaluator.evaluate(predictions)
print(f"ALS Recommendation Model RMSE: {rmse_value/1e2:.4f}")

# Show top 5 recommended stations for a few users
als_model = rec_model.stages[-1]
userRecs = als_model.recommendForAllUsers(5)
print("\nSample Route Recommendations for Passengers:")
userRecs.show(5, truncate=False)


# ==============================================================
# b. Peak Hour Prediction (Decision Tree)
# ==============================================================

print("\n==============================")
print("Training Decision Tree Classifier for Peak Hour Prediction")
print("==============================")

indexer = StringIndexer(inputCol="passenger_type", outputCol="passenger_type_index", handleInvalid="keep")
assembler = VectorAssembler(
    inputCols=["hour_of_day", "fare_amount", "travel_duration_minutes", "passenger_type_index"],
    outputCol="features"
)
dtc = DecisionTreeClassifier(featuresCol="features", labelCol="is_peak_hour", maxDepth=5)
dt_pipeline = Pipeline(stages=[indexer, assembler, dtc])

train_df, test_df = df.randomSplit([0.75, 0.25], seed=42)
dt_model = dt_pipeline.fit(train_df)
dt_predictions = dt_model.transform(test_df)

class_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_peak_hour",
    predictionCol="prediction",
    metricName="accuracy"
)
dt_accuracy = class_evaluator.evaluate(dt_predictions)
print(f"Decision Tree Accuracy (Peak Hour Classification): {dt_accuracy*100:.2f}%")

# Show sample predictions
print("\nSample Peak Hour Predictions:")
dt_predictions.select("hour_of_day", "fare_amount", "is_peak_hour", "prediction").show(10, truncate=False)


# ==============================================================
# c. Anomaly Detection (K-Means)
# ==============================================================

print("\n==============================")
print("Running K-Means Clustering for Anomaly Detection")
print("==============================")

kmeans_assembler = VectorAssembler(
    inputCols=["travel_duration_minutes", "fare_amount", "hour_of_day"],
    outputCol="features"
)
kmeans = KMeans(featuresCol="features", k=5, seed=42)
anomaly_pipeline = Pipeline(stages=[kmeans_assembler, kmeans])
anomaly_model = anomaly_pipeline.fit(df)
clustered_df = anomaly_model.transform(df)

print("Clustering completed.")

centers = anomaly_model.stages[-1].clusterCenters()

def calc_distance(features, prediction):
    center = centers[prediction]
    return float(np.linalg.norm(np.array(features) - np.array(center)))

distance_udf = udf(calc_distance, DoubleType())
clustered_df = clustered_df.withColumn("anomaly_score", distance_udf(col("features"), col("prediction")))
threshold = clustered_df.approxQuantile("anomaly_score", [0.95], 0.0)[0]
anomalies = clustered_df.filter(col("anomaly_score") >= threshold)

cluster_evaluator = ClusteringEvaluator(predictionCol="prediction", featuresCol="features", metricName="silhouette")
silhouette = cluster_evaluator.evaluate(clustered_df)

print(f"K-Means Silhouette Score: {silhouette:.2f}")
print(f"Top 5% anomalies count: {anomalies.count()}")
print("\nSample Anomalous Trips:")
anomalies.select("entry_station", "exit_station", "fare_amount", "travel_duration_minutes", "anomaly_score").show(10, truncate=False)


# ==============================================================
# d. Monte Carlo Simulation: Fare Revenue & Risk Estimation
# ==============================================================

print("\n==============================")
print("Running Monte Carlo Simulation for Fare Risk Estimation")
print("==============================")

from pyspark.sql.functions import rand

simulations = df.selectExpr("fare_amount as base_fare") \
    .sample(withReplacement=True, fraction=0.1, seed=42) \
    .limit(10000) \
    .withColumn("simulated_fare", col("base_fare") * (1 + (rand() - 0.5) * 0.2))

def compute_revenue_metrics(sim_df):
    stats = sim_df.select(
        _mean("simulated_fare").alias("mean_fare"),
        _stddev("simulated_fare").alias("stddev_fare"),
        _variance("simulated_fare").alias("variance_fare")
    ).collect()[0]

    mean_fare = stats["mean_fare"]
    std_fare = stats["stddev_fare"]
    var_fare = stats["variance_fare"]

    total_revenue = mean_fare * sim_df.count()
    lower_ci = total_revenue - (1.96 * std_fare * np.sqrt(sim_df.count()))
    upper_ci = total_revenue + (1.96 * std_fare * np.sqrt(sim_df.count()))

    print("\n==============================")
    print("MONTE CARLO REVENUE ESTIMATION RESULTS")
    print("==============================")
    print(f"Expected Mean Fare: ₹{mean_fare:.2f}")
    print(f"Standard Deviation: ₹{std_fare:.2f}")
    print(f"Variance: {var_fare:.4f}")
    print(f"Total Simulated Trips: {sim_df.count()}")
    print(f"Estimated Total Revenue: ₹{total_revenue/1e4:.3f} M")
    print(f"95% Confidence Interval: ₹{lower_ci/1e4:.3f} M – ₹{upper_ci/1e4:.3f} M")
    print(f"Relative Variance: {(var_fare/mean_fare**2)*100:.2f}%")
    print("==============================\n")

compute_revenue_metrics(simulations)

# Extended Data Analysis
print("\n==============================")
print("Extended Data Analysis")
print("==============================")

print("\nPeak Hour Travel Time by Station:")
peak_travel_time = df.filter(col("is_peak_hour") == 1) \
    .groupBy("entry_station_norm") \
    .agg(F.avg("travel_duration_minutes").alias("avg_peak_travel_duration"),
         F.count("*").alias("peak_trips_count")) \
    .orderBy(col("avg_peak_travel_duration").desc())
peak_travel_time.show(10)

print("\nOff-Peak Travel Time by Station:")
offpeak_travel_time = df.filter(col("is_peak_hour") == 0) \
    .groupBy("entry_station_norm") \
    .agg(F.avg("travel_duration_minutes").alias("avg_offpeak_travel_duration"),
         F.count("*").alias("offpeak_trips_count")) \
    .orderBy(col("avg_offpeak_travel_duration").desc())
offpeak_travel_time.show(10)

print("\nStation-wise Fare Statistics:")
fare_stats = df.groupBy("entry_station_norm") \
    .agg(F.min("fare_amount").alias("min_fare"),
         F.max("fare_amount").alias("max_fare"),
         F.mean("fare_amount").alias("avg_fare")) \
    .orderBy(col("avg_fare").desc())
fare_stats.show(10)

print("\nHourly Trip Counts:")
hourly_volume = df.groupBy("hour_of_day") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy("hour_of_day")
hourly_volume.show(24)

print("\nWeekend vs Weekday Travel Analysis:")
weekend_weekday = df.groupBy("is_weekend") \
    .agg(F.count("*").alias("total_trips"),
         F.avg("travel_duration_minutes").alias("avg_travel_duration"),
         F.avg("fare_amount").alias("avg_fare"))
weekend_weekday.show()

print("\nRepeat Travel Frequency by Card ID:")
window_spec = Window.partitionBy("card_id")
df_repeats = df.withColumn("trip_count_per_card", F.count("transaction_id").over(window_spec))
repeat_customers = df_repeats.select("card_id", "trip_count_per_card") \
    .distinct().orderBy(col("trip_count_per_card").desc())
repeat_customers.show(10)

print("\nPopular Station Pairs:")
station_pairs = df.groupBy("entry_station_norm", "exit_station_norm") \
    .agg(F.count("*").alias("trip_count"),
         F.avg("fare_amount").alias("avg_fare")) \
    .orderBy(col("trip_count").desc())
station_pairs.show(10)

print("\nPassenger Type Distribution per Station:")
passenger_type_dist = df.groupBy("entry_station_norm", "passenger_type") \
    .agg(F.count("*").alias("count")) \
    .orderBy("entry_station_norm", "passenger_type")
passenger_type_dist.show(20)

print("\nFare Amount Outlier Detection:")
fare_stats_overall = df.select(
    F.mean("fare_amount").alias("mean_fare"),
    F.stddev("fare_amount").alias("stddev_fare")
).collect()[0]
upper_threshold = fare_stats_overall["mean_fare"] + 3 * fare_stats_overall["stddev_fare"]
outliers = df.filter(col("fare_amount") > upper_threshold)
print(f"Number of fare outliers: {outliers.count()}")

# Save processed data as Parquet
print("\n==============================")
print("Saving Processed Data as Parquet")
print("==============================")
df.write.mode("overwrite").parquet("delhi_metro_processed_parquet")
print("Data successfully saved to 'delhi_metro_processed_parquet'")

spark.stop()
print("\n==============================")
print("Spark Session Stopped. All Tasks Completed Successfully!")
print("==============================")



Initializing Spark Session

Loading CSV Data...
Data Loaded Successfully.
Total Records: 20000
Columns: ['transaction_id', 'card_id', 'entry_station', 'exit_station', 'entry_time', 'exit_time', 'travel_duration_minutes', 'fare_amount', 'card_balance', 'passenger_type', 'metro_line']

Data Preprocessing & Cleansing

Data Schema:
root
 |-- transaction_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- entry_station: string (nullable = true)
 |-- exit_station: string (nullable = true)
 |-- entry_time: timestamp (nullable = true)
 |-- exit_time: timestamp (nullable = true)
 |-- travel_duration_minutes: integer (nullable = true)
 |-- fare_amount: integer (nullable = true)
 |-- card_balance: integer (nullable = true)
 |-- passenger_type: string (nullable = true)
 |-- metro_line: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_peak_hour: integer (nullable = false)
 |-- is_weekend: integer (nullable

+---------+----------------------------------------------------------------------------------+
|userIndex|recommendations                                                                   |
+---------+----------------------------------------------------------------------------------+
|12       |[{15, 14.791831}, {6, 13.910049}, {10, 13.716451}, {0, 10.526054}, {13, 9.634644}]|
|22       |[{5, 17.93871}, {12, 17.921892}, {1, 11.247164}, {3, 9.611228}, {11, 9.011471}]   |
|26       |[{4, 53.64807}, {0, 51.728786}, {5, 20.53056}, {3, 19.91031}, {11, 19.703281}]    |
|27       |[{16, 57.019424}, {17, 46.424583}, {9, 40.809723}, {11, 33.28047}, {8, 26.418966}]|
|28       |[{5, 53.868233}, {11, 43.28555}, {3, 38.471756}, {9, 34.70926}, {17, 28.706871}]  |
+---------+----------------------------------------------------------------------------------+
only showing top 5 rows

Training Decision Tree Classifier for Peak Hour Prediction
Decision Tree Accuracy (Peak Hour Classification): 100.00%

S

K-Means Silhouette Score: 0.52
Top 5% anomalies count: 1001

Sample Anomalous Trips:


Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/Users/pranavkumar/Desktop/Coding/Python/BDA/.pyspark-env/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


+----------------+---------------+-----------+-----------------------+------------------+
|entry_station   |exit_station   |fare_amount|travel_duration_minutes|anomaly_score     |
+----------------+---------------+-----------+-----------------------+------------------+
|Airport         |Noida Sector 62|32         |78                     |19.71085619039429 |
|Botanical Garden|INA            |15         |80                     |19.135347989349025|
|Vaishali        |INA            |29         |10                     |19.676222472132626|
|Vaishali        |Kashmere Gate  |31         |13                     |18.063084196815687|
|Noida Sector 62 |Kashmere Gate  |10         |16                     |19.80874667985345 |
|Airport         |Connaught Place|15         |46                     |18.732800724215682|
|Noida Sector 62 |Dwarka         |19         |45                     |18.248632232910765|
|Green Park      |Anand Vihar    |18         |78                     |18.9566848742556  |
|Chandni C